# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect the data, check for leakage and censoring, then evaluate honest models with a **time-based** split (train on earlier inspections, test on later ones), as a deployed model would be used.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'missing:', int(df.isna().sum().sum()), '| unique assets:', df.asset_tag.nunique())
print('base rate:', df.failed_90d.mean().round(4))
df.describe().T

## 1. Leakage check: `parts_replaced`
Per the dictionary, this is the parts count on the work order that *closed* the inspection record, i.e. it is only known after the work is done, possibly after the failure.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','size'])

Every record with 2+ parts replaced failed, and none with 0 did. That is almost a perfect proxy for the label, so it is probably the repair that followed a failure. It would not be available at inspection time, so **I exclude it** from the models. One model below includes it only to show how inflated the score gets.

## 2. Time trend and possible right-censoring

In [ ]:
q = df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.agg(['mean','size'])
q['mean'].plot(marker='o', figsize=(9,3), title='Failure rate by inspection month'); plt.ylabel('rate'); plt.show()
print('Data ends:', df.inspected_at.max())

The failure rate swings over time. Inspections in the final 90 days before the end of the data cannot be fully observed (a failure after the last date would be recorded as 0), so part of any late drop may be an artifact. I drop the last 90 days of inspections from the modelling data.

In [ ]:
cutoff = df.inspected_at.max() - pd.Timedelta(days=90)
d = df[df.inspected_at <= cutoff].reset_index(drop=True)
print(len(df)-len(d), 'rows dropped (censored window); remaining', len(d))
d.groupby(d.inspected_at.dt.to_period('Q')).failed_90d.mean().round(3)

## 3. Feature relationships

In [ ]:
num = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig, ax = plt.subplots(1,5, figsize=(16,3))
for a,c in zip(ax,num):
    b = pd.qcut(d[c], 10, duplicates='drop')
    d.groupby(b, observed=True).failed_90d.mean().reset_index(drop=True).plot(ax=a, marker='o'); a.set_title(c); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
print(d.groupby('site').failed_90d.mean().round(3))

## 4. Models
Features known at inspection time: sensor readings and site (and, in one variant, calendar month). Evaluation is **expanding-window time-series CV**: each fold is trained only on earlier inspections. Metrics: ROC-AUC and average precision (chance level = base rate).

In [ ]:
d['month'] = d.inspected_at.dt.month
base_feats = num + ['site']
def pipe(model, feats):
    pre = ColumnTransformer([('n', StandardScaler(), [f for f in feats if f!='site']),
                             ('c', OneHotEncoder(), ['site'])])
    return make_pipeline(pre, model)
gb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
tscv = TimeSeriesSplit(n_splits=5)
def evaluate(model, feats):
    X, y = d[feats], d.failed_90d.values
    rows=[]
    for tr, te in tscv.split(X):
        m = model.fit(X.iloc[tr], y[tr]); p = m.predict_proba(X.iloc[te])[:,1]
        rows.append((roc_auc_score(y[te],p), average_precision_score(y[te],p), y[te].mean()))
    return pd.DataFrame(rows, columns=['auc','ap','base_rate'])
res = {}
res['logreg (no leak)']  = evaluate(pipe(LogisticRegression(max_iter=1000), base_feats), base_feats)
res['hist-GB (no leak)'] = evaluate(pipe(gb(), base_feats), base_feats)
res['hist-GB + month']   = evaluate(pipe(gb(), base_feats+['month']), base_feats+['month'])
res['hist-GB + parts_replaced (LEAKY)'] = evaluate(pipe(gb(), base_feats+['parts_replaced']), base_feats+['parts_replaced'])
summary = pd.DataFrame({k:v.mean() for k,v in res.items()}).T.round(3)
summary['auc_std'] = [v.auc.std().round(3) for v in res.values()]
summary

In [ ]:
res['logreg (no leak)'].round(3)

## 5. Final check: hold out the most recent 20% of inspections

In [ ]:
cut = int(len(d)*0.8); tr, te = d.iloc[:cut], d.iloc[cut:]
m = pipe(LogisticRegression(max_iter=1000), base_feats).fit(tr[base_feats], tr.failed_90d)
p = m.predict_proba(te[base_feats])[:,1]
print(f'test n={len(te)}, base rate={te.failed_90d.mean():.3f}')
print(f'AUC={roc_auc_score(te.failed_90d,p):.3f}  AP={average_precision_score(te.failed_90d,p):.3f}')
pd.Series(m[-1].coef_[0], index=m[0].get_feature_names_out()).sort_values().round(3)

In [ ]:
te = te.assign(p=p); te['risk_quintile'] = pd.qcut(te.p, 5, labels=False, duplicates='drop')
te.groupby('risk_quintile').failed_90d.agg(['mean','size']).rename(columns={'mean':'observed failure rate'})

## Conclusions
(Numbers from running this notebook's code; see the tables above.)

- **Yes, but only modestly.** With leak-free features and a time-based split, ROC-AUC is about 0.70 to 0.74 (logistic regression 0.71, gradient boosting 0.74), against 0.5 for chance. Average precision is about 0.30 to 0.32 versus a 15% base rate, so the model roughly doubles precision over guessing. Fold-to-fold AUC varies a lot (0.56 to 0.78), and the failure rate itself moves a lot over time.
- **`parts_replaced` is leakage.** Including it gives AUC about 0.997, but it is recorded on the work order that closes the record, so it reflects the repair after a failure. It must not be used for prediction.
- **Signal sits in vibration, load and bearing temperature** (higher means riskier). Ambient temperature and meter hours have negative coefficients. That is unexpected and may reflect time and seasonal confounding rather than a causal effect, so treat it with caution. Site carries essentially no information.
- **The most recent 20% holdout is weaker** (AUC 0.68, AP 0.14 on a 7.6% base rate). Risk ranking still works: the top quintile fails at about 14% and the bottom at about 2%. The drift in base rate means probabilities would need recalibration.
- **Censoring:** inspections within 90 days of the end of the data were dropped, because their label is unreliable.

**Caveats / next steps:** each asset appears once, so there is no per-asset history; adding prior inspections (trends in vibration, time since last failure) is the most likely route to better accuracy. The roughly 12% base rate and the large swings by quarter suggest checking how failures are recorded, and whether a process or seasonal change explains the mid-2024 spike. Pick an alert threshold using the cost of a missed failure versus a false alarm.